# About this notebook

In this notebook, we download the latest PDFs directly from TOME's owncloud.

In [1]:
import fitz
import requests
import io
import pickle
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import re
from bs4 import BeautifulSoup
import pandas as pd
import nltk
#!pip install sddk
import sddk # our package, if missing, uncomment the previous line
import json
import requests
import getpass
from bs4 import BeautifulSoup
from requests_oauthlib import OAuth1
import zipfile
import io
import os
import re
import tempfile
import zipfile
from pathlib import Path
import stat
import posixpath
from pathlib import Path
import zipfile
import shutil

In [5]:
creds = json.load(open("../owncloud_creds.json", "r"))

In [6]:
# accessing owncloud.cesnet.cz with sddk package
user = creds["user"] # input("Insert your Username code (a long string of characters and numbers): ")
password = creds["password"] # getpass.getpass("Insert your Password: ")
s = requests.Session() # create session
s.auth = (user, password)

In [7]:
resp = s.get("https://owncloud.cesnet.cz/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317181.zip")
resp

<Response [200]>

In [12]:
with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        # List the contents of the ZIP file
        print("Files in the ZIP archive:")
        print(z.namelist())

        # Extract contents to a folder (e.g., "./extracted_files")
        extract_dir = "../data/test_pdf+xml/"
        print(f"Extracting files to: {extract_dir}")
        z.extractall(extract_dir)
        print("Extraction complete!")


Files in the ZIP archive:
['/log.txt', '1698730/Dorn1569_Artificii_chymistici_MDZ_MBS.pdf', '1698730/Dorn1569_Artificii_chymistici_MDZ_MBS/metadata.xml']
Extracting files to: ../data/test_pdf+xml/
Extraction complete!


In [37]:
# 75(76) files from April
base_url = "https://owncloud.cesnet.cz/"
resp = s.request("PROPFIND", base_url + "/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Standard%20Export")
resp

<Response [207]>

In [39]:
resp.text

'<?xml version="1.0"?>\n<d:multistatus xmlns:d="DAV:" xmlns:s="http://sabredav.org/ns" xmlns:oc="http://owncloud.org/ns"><d:response><d:href>/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Standard%20Export/</d:href><d:propstat><d:prop><d:getlastmodified>Mon, 27 Oct 2025 13:28:55 GMT</d:getlastmodified><d:resourcetype><d:collection/></d:resourcetype><d:quota-used-bytes>75887872589</d:quota-used-bytes><d:quota-available-bytes>71448213894</d:quota-available-bytes><d:getetag>&quot;68ff739790852&quot;</d:getetag></d:prop><d:status>HTTP/1.1 200 OK</d:status></d:propstat></d:response><d:response><d:href>/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Standard%20Export/export_job_18762445.zip</d:href><d:propstat><d:prop><d:getlastmodified>Thu, 09 Oct 2025 09:36:47 GMT</d:getlastmodified><d:getcontentlength>1236447877</d:getcontentlength><d:resourcetype/><d:getetag>&quot;728f7d8eb9b2cd5dcda1979efcf97d66&quo

In [40]:
soup = BeautifulSoup(resp.text, "xml")
all_items = soup.find_all("d:response")

In [41]:
len(all_items)

24

In [42]:
# urls of individual zip files from "Standard Export" folder
hrefs_standard = []
for item in all_items:
     href = item.find("d:href").text
     hrefs_standard.append(href)
hrefs_standard

['/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Standard%20Export/',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Standard%20Export/export_job_18762445.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Standard%20Export/export_job_18763073.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Standard%20Export/export_job_18764028.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Standard%20Export/export_job_18765631.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Standard%20Export/export_job_18769920.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Standard%20Export/export_job_18771003.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5

In [43]:
# Alto XML from October
base_url = "https://owncloud.cesnet.cz/"
resp = s.request("PROPFIND", base_url + "/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Alto%20XML")
resp

<Response [207]>

In [44]:
soup = BeautifulSoup(resp.text, "xml")
all_items = soup.find_all("d:response")

In [45]:
# urls of individual zip files from "Standard Export" folder
hrefs_alto = []
for item in all_items:
     href = item.find("d:href").text
     hrefs_alto.append(href)
hrefs_alto

['/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Alto%20XML/',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Alto%20XML/export_job_18762460.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Alto%20XML/export_job_18763082.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Alto%20XML/export_job_18764041.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Alto%20XML/export_job_18765643.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Alto%20XML/export_job_18769928.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Alto%20XML/export_job_18771019.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Alto%20XML/e

In [30]:
target_dir = "/srv/data/tome/tome-corpus/EMLAP_2025-10-31"
try:
    os.mkdir(target_dir)
except FileExistsError:
    print("Directory already exists.")

In [31]:
os.listdir(target_dir)

[]

In [33]:
resp = s.get(base_url + hrefs_standard[1])

In [52]:
os.makedirs("/srv/data/tome/tome-corpus/EMLAP_2025-10-31/standard", exist_ok=True)
os.makedirs("/srv/data/tome/tome-corpus/EMLAP_2025-10-31/alto", exist_ok=True)

In [51]:
# Assumes you already have: s = requests.Session()
base_url = "https://owncloud.cesnet.cz/"

CHUNK = 16 * 1024 * 1024  # 16 MB chunks: good for large files

def _sanitize_zip_path(name: str) -> str | None:
    """
    Return a safe, normalized, relative path for a Zip member.
    - Strips leading slashes
    - Normalizes path (no ../ escapes)
    - Drops Windows drive letters ("C:")
    - Converts backslashes to forward slashes
    Returns None if the path would escape the destination.
    """
    # Normalize separators and strip Windows drive letters like "C:" or "D:"
    name = name.replace("\\", "/")
    if ":" in name:
        # keep only the part after the colon (drop drive)
        name = name.split(":", 1)[1]

    # Remove any leading slashes so "/log.txt" -> "log.txt"
    name = name.lstrip("/")

    # Collapse things like "a/../b"
    norm = posixpath.normpath(name)

    # After normpath, block anything that would climb out
    if norm == ".." or norm.startswith("../"):
        return None

    return norm

def safe_extract_streaming(zf: zipfile.ZipFile, dest_dir: Path, verbose: bool = True):
    """
    Extracts members one-by-one with path sanitization and streaming I/O.
    Skips symlinks and unsafe paths.
    """
    dest_dir = Path(dest_dir)

    for info in zf.infolist():
        # Directories are easy — just ensure existence
        is_dir = info.is_dir() if hasattr(info, "is_dir") else info.filename.endswith("/")
        safe_rel = _sanitize_zip_path(info.filename)
        if safe_rel is None:
            if verbose:
                print(f" [skip] unsafe path in ZIP: {info.filename!r}")
            continue

        target_path = dest_dir / safe_rel

        # Ensure the target stays inside dest_dir
        try:
            if not str(target_path.resolve()).startswith(str(dest_dir.resolve())):
                if verbose:
                    print(f" [skip] resolved outside: {info.filename!r}")
                continue
        except FileNotFoundError:
            # parent dirs may not exist yet; that's fine
            pass

        # Detect *nix symlinks via external attributes; skip them for safety
        is_symlink = (info.external_attr >> 16) & stat.S_IFMT(stat.S_IFLNK) == stat.S_IFLNK
        if is_symlink:
            if verbose:
                print(f" [skip] symlink in ZIP: {info.filename!r}")
            continue

        if is_dir:
            target_path.mkdir(parents=True, exist_ok=True)
            continue

        # Ensure parent directory exists
        target_path.parent.mkdir(parents=True, exist_ok=True)

        # Stream file content out
        with zf.open(info, "r") as src, open(target_path, "wb") as dst:
            while True:
                chunk = src.read(16 * 1024 * 1024)  # 16 MB
                if not chunk:
                    break
                dst.write(chunk)

        # (Optional) restore executable bit (basic)
        mode = (info.external_attr >> 16) & 0o777
        if mode:
            try:
                os.chmod(target_path, mode)
            except Exception:
                pass
        if verbose:
            print(f" [ok] {safe_rel}")

def download_with_resume(session, url, dest_path):
    """
    Download to dest_path with HTTP Range resume support.
    Returns the final file path.
    """
    dest_path = Path(dest_path)
    # Write to a temp file in the same directory for atomic move
    tmp_fd, tmp_name = tempfile.mkstemp(prefix=".part_", dir=str(dest_path.parent))
    os.close(tmp_fd)
    tmp_path = Path(tmp_name)

    # If a previous partial exists, keep using it
    existing = tmp_path.stat().st_size if tmp_path.exists() else 0

    headers = {}
    if existing > 0:
        headers["Range"] = f"bytes={existing}-"

    # HEAD to learn total size (optional)
    try:
        head = session.head(url, allow_redirects=True, timeout=30)
        head.raise_for_status()
        total = int(head.headers.get("Content-Length", "0"))
    except Exception:
        total = 0  # not critical

    with session.get(url, stream=True, headers=headers, timeout=60) as r:
        if r.status_code not in (200, 206):
            r.raise_for_status()

        # If server ignored Range, restart
        if r.status_code == 200 and existing:
            existing = 0  # overwrite
            tmp_path.unlink(missing_ok=True)

        mode = "ab" if existing else "wb"
        downloaded = existing

        with open(tmp_path, mode) as f:
            for chunk in r.iter_content(chunk_size=CHUNK):
                if chunk:  # keep-alive chunks may be empty
                    f.write(chunk)
                    downloaded += len(chunk)
                    # Light progress (avoids extra deps)
                    if total:
                        pct = (downloaded / (existing + total)) * 100
                        print(f"\rDownloading {dest_path.name}: {pct:5.1f}% ({downloaded/1e6:,.0f} MB)", end="")
                    else:
                        print(f"\rDownloading {dest_path.name}: {downloaded/1e6:,.0f} MB", end="")

    print()  # newline after progress
    # Move into place atomically
    tmp_path.replace(dest_path)
    return dest_path

In [ ]:
extract_root = Path("/srv/data/tome/tome-corpus/EMLAP_2025-10-31/standard/")

for href in hrefs_standard:
    if href.lower().endswith(".zip"):
        url = base_url + href.lstrip("/")
        zip_name = Path(href).name
        # Extract each ZIP into its own subfolder (based on filename without .zip)
        target_dir = extract_root / Path(zip_name).stem
        target_dir.mkdir(parents=True, exist_ok=True)
        local_zip = target_dir.with_suffix(".zip")  # e.g., foo/foo.zip

        try:
            print(f"{href} - starting download (resumable)…")
            download_with_resume(s, url, local_zip)

            # allowZip64=True handles >4GB archives (default is True)
            with zipfile.ZipFile(local_zip, "r", allowZip64=True) as z:
                safe_extract_streaming(z, target_dir, verbose=True)
            print(f"{href} - extraction complete to {target_dir}")

            # Optional cleanup to save disk space:
            try:
                local_zip.unlink()
                print(f"{href} - removed {local_zip.name}")
            except Exception as cleanup_err:
                print(f"{href} - could not delete ZIP: {cleanup_err}")

        except Exception as e:
            print(f"{href} - failed: {e}")

In [54]:
extract_root = Path("/srv/data/tome/tome-corpus/EMLAP_2025-10-31/alto/")

for href in hrefs_alto:
    if href.lower().endswith(".zip"):
        url = base_url + href.lstrip("/")
        zip_name = Path(href).name
        # Extract each ZIP into its own subfolder (based on filename without .zip)
        target_dir = extract_root / Path(zip_name).stem
        target_dir.mkdir(parents=True, exist_ok=True)
        local_zip = target_dir.with_suffix(".zip")  # e.g., foo/foo.zip

        try:
            print(f"{href} - starting download (resumable)…")
            download_with_resume(s, url, local_zip)

            # allowZip64=True handles >4GB archives (default is True)
            with zipfile.ZipFile(local_zip, "r", allowZip64=True) as z:
                safe_extract_streaming(z, target_dir, verbose=True)

            print(f"{href} - extraction complete to {target_dir}")

            # Optional cleanup to save disk space:
            try:
                local_zip.unlink()
                print(f"{href} - removed {local_zip.name}")
            except Exception as cleanup_err:
                print(f"{href} - could not delete ZIP: {cleanup_err}")

        except Exception as e:
            print(f"{href} - failed: {e}")

/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw%20v2/Alto%20XML/export_job_18762460.zip - starting download (resumable)…
 [ok] 9676157/100086_Penotus1608_Denario_medico_MDZ_MBS/metadata.xml
 [ok] 9676157/100086_Penotus1608_Denario_medico_MDZ_MBS/alto/0035_p035.xml
 [ok] 9676157/100086_Penotus1608_Denario_medico_MDZ_MBS/alto/0002_p002.xml
 [ok] 9676157/100086_Penotus1608_Denario_medico_MDZ_MBS/alto/0148_p148.xml
 [ok] 9676157/100086_Penotus1608_Denario_medico_MDZ_MBS/alto/0055_p055.xml
 [ok] 9676157/100086_Penotus1608_Denario_medico_MDZ_MBS/alto/0197_p197.xml
 [ok] 9676157/100086_Penotus1608_Denario_medico_MDZ_MBS/alto/0033_p033.xml
 [ok] 9676157/100086_Penotus1608_Denario_medico_MDZ_MBS/alto/0001_p001.xml
 [ok] 9676157/100086_Penotus1608_Denario_medico_MDZ_MBS/alto/0049_p049.xml
 [ok] 9676157/100086_Penotus1608_Denario_medico_MDZ_MBS/alto/0087_p087.xml
 [ok] 9676157/100086_Penotus1608_Denario_medico_MDZ_MBS/alto/0103_p103.xml
 [ok] 9676157/100086_

In [4]:
# copy pdfs into a new independent directory.
extract_root = Path("/srv/data/tome/tome-corpus/EMLAP_2025-10-31/standard/")
pdfs_dir = "/srv/data/tome/tome-corpus/EMLAP_2025-10-31/pdfs_only/"
os.makedirs(pdfs_dir, exist_ok=True)

In [5]:
for path in extract_root.rglob("*"):
    if path.is_file() and path.suffix.lower() == ".pdf":
        dest = pdfs_dir + path.name
        shutil.copy2(path, dest)

In [7]:
os.listdir(pdfs_dir)

['100012_Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ.pdf',
 '100090_Nolle1617_Theoria_philosophiae_Hermeticae_MDZ_MBS.pdf',
 '100067_Wecker1582_De_secretis_ONB.pdf',
 '100014_Toxites1567_Spongia_stibii_MDZ_MBS.pdf',
 '100072_Andernach1571_De_medicina_veteri_et_novi_MDZ_MBS.pdf',
 '100024_Fanianus1560_De_arte_metallicae_ONB.pdf',
 '100055_Erastus1578_Disputatio_de_auro_potabili_MDZ_MBS.pdf',
 '100083_Sendivogius1616_Tractatus_de_sulphure_MBS_MDZ.pdf',
 '100036_Vadis1595_Dialogus_IA_Wellcome.pdf',
 '100094_Anon1625_Musaeum_hermeticum_VD17_SLUB.pdf',
 '100019_Ulstadt1526_De_epidemia_ONB.pdf',
 '100032_Ventura1571_De_ratione_conficiendi_lapis_MBZ_MBS.pdf',
 '100016_Bonus1546_Pretiosa_Margarita_Novella_ONB.pdf',
 '100008_Severinus1572_Epistola_MBZ_MBS.pdf',
 '100034_Penotus1594_Tractatus_varii_MDZ_MBS.pdf',
 '100069_Severinus1571_Idea_medicinae_philosophicae_GB_Noscemus.pdf',
 '100020_Dorn1569_Artificii_chymistici_MDZ_MBS.pdf',
 '100062_Albertus1569_De_concordantia_Hippocraticorum_et_Para

In [8]:
len(os.listdir(pdfs_dir))

100

In [ ]:
unzipped_dirs = os.listdir()
for dir in os.listdir("/srv/data/tome/tome-corpus/emlap_raw_2025-04-08/"):
    filename = [f for f in os.listdir(os.path.join(source_dir, dir)) if ".pdf" in f][0]
    filepath = os.path.join(source_dir, dir, filename)
    # copy file to